# Fake Text-Completion Models

The `fake.py` module defines deterministic text-completion language models for testing LangChain applications.

`FakeListLLM` returns configured string responses in cyclic order. `FakeStreamingListLLM` extends it with synchronous and asynchronous character streaming, optional delays between chunks, and configurable injected failures.

# FakeListLLM: `LLM`

`FakeListLLM` is a fake text-completion model that returns responses from a configured list in sequence.

After the final response is returned, its internal index resets to zero so later invocations begin again with the first response.

## Fields

1. `responses`:`list[str]`:= Stores the string responses returned by successive model invocations.

   Responses are selected in order and repeated from the beginning after the final entry.

2. `sleep`:`float | None`:= Stores an optional delay in seconds for subclasses. Its default value is `None`.

   `FakeListLLM` does not use this field directly. `FakeStreamingListLLM` uses it between streamed characters.

3. `i`:`int`:= Stores the index of the response returned by the next invocation. Its default value is `0`.

   The index advances after each synchronous or asynchronous call and resets to zero after the final configured response.

## Properties

1. `_llm_type`:`str`:= Returns the identifying model type. Its value is `"fake-list"`.

2. `_identifying_params`:`Mapping[str, Any]`:= Returns the configured responses as the model's identifying parameters.

   **Syntax**

   ```python
   {
       "responses": self.responses
   }
   ```

## Methods

1. `_call`:= Returns the next configured response synchronously.

   The prompt, stop sequences, callback manager, and additional keyword arguments do not affect response selection. The internal index advances after the response is selected and wraps to zero after the final response.

   **Syntax**

   ```python
   _call(
       self, # Fake list LLM instance
       prompt: str, # Input prompt
       stop: list[str] | None = None, # Optional stop sequences
       run_manager: CallbackManagerForLLMRun | None = None, # Optional callback manager
       **kwargs: Any # Additional generation parameters
   ) -> str
   ```

2. `_acall`:= Returns the next configured response asynchronously.

   No external asynchronous operation is performed. Response selection and index rotation follow the same rules as `_call()`.

   **Syntax**

   ```python
   async _acall(
       self, # Fake list LLM instance
       prompt: str, # Input prompt
       stop: list[str] | None = None, # Optional stop sequences
       run_manager: AsyncCallbackManagerForLLMRun | None = None, # Optional asynchronous callback manager
       **kwargs: Any # Additional generation parameters
   ) -> str
   ```

# FakeListLLMError: `Exception`

`FakeListLLMError` represents the deliberate streaming failure raised by `FakeStreamingListLLM`.

It is raised when streaming reaches the configured `error_on_chunk_number`.



In [ ]:
from langchain_core.language_models.fake import FakeListLLM, FakeStreamingListLLM, FakeListLLMError # Import the fake LLM classes and streaming error

model = FakeListLLM( # Create a cyclic fake text-completion model
    responses=["First response", "Second response"] # Define responses returned sequentially
) # Finish creating the model

print(model.invoke("Prompt 1")) # Return and display the first response
print(await model.ainvoke("Prompt 2")) # Return and display the second response asynchronously
print(model.invoke("Prompt 3")) # Cycle back and display the first response again

streaming_model = FakeStreamingListLLM( # Create a character-streaming fake model
    responses=["Hello"], # Define the response to stream
    sleep=None # Disable delay between streamed characters
) # Finish creating the streaming model

for chunk in streaming_model.stream("Stream this"): # Stream the response synchronously
    print(chunk, end="") # Display each character without starting a new line

print() # Move to the next output line

async for chunk in streaming_model.astream("Stream asynchronously"): # Stream the response asynchronously
    print(chunk, end="") # Display each asynchronous character chunk

print() # Move to the next output line

error_model = FakeStreamingListLLM( # Create a model configured to fail during streaming
    responses=["Python"], # Define the response to stream
    error_on_chunk_number=2 # Raise an error before yielding character index 2
) # Finish creating the error-producing model

try: # Handle the expected streaming failure
    for chunk in error_model.stream("Test failure"): # Begin streaming the configured response
        print(chunk, end="") # Display characters produced before the failure
except FakeListLLMError: # Catch the deliberate streaming error
    print("\nFakeListLLMError was raised.") # Confirm that the configured failure occurred

# FakeStreamingListLLM: `FakeListLLM`

`FakeStreamingListLLM` extends `FakeListLLM` with character-by-character synchronous and asynchronous streaming.

Each stream first obtains one complete cyclic response through normal invocation and then yields its characters individually.

## Fields

1. `error_on_chunk_number`:`int | None`:= Stores the zero-based character index at which streaming raises `FakeListLLMError`. Its default value is `None`.

   A value of `None` disables injected failures. The exception is raised before the character at the configured index is yielded.

## Methods

1. `stream`:= Invokes the model synchronously and yields the selected response one character at a time.

   When `sleep` is configured, `time.sleep()` runs before each character. When the current zero-based character index matches `error_on_chunk_number`, `FakeListLLMError` is raised instead of yielding that character.

   The explicit `stop` parameter and additional keyword arguments are accepted by this override but are not forwarded to `invoke()`.

   **Syntax**

   ```python
   stream(
       self, # Fake streaming LLM instance
       input: LanguageModelInput, # Input accepted by the language model
       config: RunnableConfig | None = None, # Runnable configuration
       *,
       stop: list[str] | None = None, # Optional stop sequences
       **kwargs: Any # Additional streaming parameters
   ) -> Iterator[str]
   ```

2. `astream`:= Invokes the model asynchronously and yields the selected response one character at a time.

   When `sleep` is configured, `asyncio.sleep()` is awaited before each character. Injected failure behaviour matches `stream()`.

   The explicit `stop` parameter and additional keyword arguments are accepted by this override but are not forwarded to `ainvoke()`.

   **Syntax**

   ```python
   async astream(
       self, # Fake streaming LLM instance
       input: LanguageModelInput, # Input accepted by the language model
       config: RunnableConfig | None = None, # Runnable configuration
       *,
       stop: list[str] | None = None, # Optional stop sequences
       **kwargs: Any # Additional streaming parameters
   ) -> AsyncIterator[str]
   ```